<div dir=rtl style="text-align: right">

# שיעור 3 — Backpropagation: איך רשת נוירונים לומדת?

## המוטיבציה

בשיעורים הקודמים ראינו:
- רשת נוירונים היא **פונקציה פרמטרית**
- Gradient Descent מוצא פרמטרים שממזערים את ה-Loss
- לשם כך אנחנו צריכים **גרדיאנטים** — כמה כל פרמטר משפיע על ה-Loss

**השאלה:** איך מחשבים גרדיאנטים לגבי **כל** הפרמטרים בצורה יעילה?

בדוגמת השיעור הראשון היו **2 פרמטרים** (a, b) — גזרנו אנליטית ביד.  
ב-GPT-3 יש **175 מיליארד פרמטרים**. זה לא אפשרי ביד.

**Backpropagation** הוא האלגוריתם שפותר את הבעיה הזו — מחשב **את כל הגרדיאנטים בפעם אחת**, ביעילות, לכל רשת כמעט שנרצה לבנות.

הרעיון המרכזי: **חוק השרשרת**, מיושם חכם.

</div>

<div dir=rtl style="text-align: right">

## רקע היסטורי — איך הגענו לכאן?

### חוק השרשרת: מאות שנים לפני הבינה המלאכותית

חוק השרשרת עצמו הוא מתמטיקה קלאסית — **לייבניץ** ו**ניוטון** פיתחו אותו כבר במאה ה-17 כחלק מהחדו"א.  
אבל היישום שלו לאימון רשתות נוירונים לקח עוד שלוש מאות שנה.

---

### 1960–1970: הרעיון מופיע — ואף אחד לא שם לב

**פול ורבוס (Paul Werbos)**, בדוקטורט שלו בהרווארד (1974), תיאר לראשונה את backpropagation כשיטה לאימון רשתות נוירונים.  
הוא כתב זאת בנספח — ועולם האקדמיה כמעט התעלם לחלוטין.

לפניו, **ספו לינאינמאה (Seppo Linnainmaa)**, מתמטיקאי פיני, פיתח בעצמאות מלאה בעבודת המאסטר שלו (1970) אלגוריתם זהה לחישוב נגזרות אוטומטי — מה שהיום קוראים **Automatic Differentiation**.  
גם הוא נשכח.

---

### 1986: הנייר שהחליף הכל

**דייויד רומלהארט (David Rumelhart), ג'ף הינטון (Geoffrey Hinton), ורונלד וויליאמס (Ronald Williams)** פרסמו בספטמבר 1986 מאמר בכתב העת *Nature*:

> **"Learning representations by back-propagating errors"**

זה לא היה הניסוח הראשון של האלגוריתם — אבל זה היה הפעם הראשונה שהוא הוצג בצורה ברורה, עם ניסויים משכנעים, לקהל רחב.  
המאמר הזה הפך ל-**אחד הנייר המצוטטים ביותר בהיסטוריה של המדעים**.

> "We describe a new learning procedure, back-propagation, for networks of neurone-like units."  
> — Rumelhart, Hinton, Williams, 1986

---

### 1986–2010: "חורף הבינה המלאכותית"

למרות הפריצה, backprop לבד לא הספיק — הרשתות היו רדודות מדי, המחשבים חלשים מדי, והדאטה מועט.  
הינטון ועמיתיו המשיכו לחקור לבד, כמעט ללא מימון, בזמן שרוב הקהילה עברה ל-SVM ושיטות קלאסיות.

---

### 2012: רגע ה"ביג בנג" של הדיפ לרנינג

**AlexNet** — רשת עמוקה שאימנה אלכס קריז'בסקי (תלמיד של הינטון) על GPU —  
ניצחה את כל המתחרים בתחרות ImageNet ב-2012 בפער עצום.  
מאז, כל התעשייה הבינה: **Backprop + GPUs + Big Data = עובד**.

הינטון קיבל על עבודתו פרס נובל לפיזיקה ב-**2024**, יחד עם **ג'ון הופילד (John Hopfield)**.

---

### נקודה מרתקת לסיום

כיום, אף אחד לא כותב backprop ביד — **PyTorch ו-TensorFlow עושים זאת אוטומטית** (autograd).  
כל פעולה מגדירה forward ו-backward, והמסגרת בונה את הגרף ומפעילה את חוק השרשרת אוטומטית.  
הרעיון של ורבוס (1974) ולינאינמאה (1970) הפך לבסיס של כל הבינה המלאכותית המודרנית.

</div>"
  }
 ],

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from ipywidgets import interact, FloatSlider, Layout

# suppress "Glyph missing from font" warnings (Hebrew chars in matplotlib)
warnings.filterwarnings('ignore', message='Glyph.*missing from font')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

<div dir=rtl style="text-align: right">

## הבעיה: גרדיאנטים לכמות עצומה של פרמטרים

כדי להבין למה אנחנו צריכים אלגוריתם, בואו נסתכל על קנה מידה.

</div>

In [ ]:
# גודל רשתות שונות
networks = [
    ('שיעור 1 שלנו (פרבולה)',          2),
    ('רשת קטנה: 2 שכבות, 64 נוירונים',  1*64 + 64 + 64*64 + 64 + 64*1 + 1),
    ('ResNet-50 (סיווג תמונות)',          25_000_000),
    ('BERT-base (NLP)',                  110_000_000),
    ('GPT-3',                           175_000_000_000),
]

print(f"{'מודל':<42} {'פרמטרים':>18}  {'גרדיאנטים לחישוב'}")
print('-' * 80)
for name, n in networks:
    method = 'גזירה ידנית אפשרית' if n <= 10 else ('... חישוב נומרי יקר מאוד' if n < 1_000_000 else 'backprop בלבד!')
    print(f"{name:<42} {n:>18,}  {method}")

<div dir=rtl style="text-align: right">

## הבסיס המתמטי: חוק השרשרת

Backpropagation הוא בסך הכל **חוק השרשרת** (Chain Rule) מחדו"א, מיושם בצורה חכמה.

**חוק השרשרת:** אם $y = f(g(x))$, אז:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} \quad \text{כאשר } u = g(x)$$

הגרדיאנט **מתרכב** — הוא מכפלת הגרדיאנטים של כל הפונקציות בשרשרת.

**רשת נוירונים = הרכבת פונקציות:**
$$L = \text{Loss}(\hat{y}(h(z(x))))$$

חוק השרשרת מאפשר לנו לחשב $\frac{\partial L}{\partial w}$ **לכל** פרמטר $w$ בשרשרת, על ידי **הכפלת הגרדיאנטים לאחור**.

</div>

In [ ]:
# חוק השרשרת — דוגמה קונקרטית
# y = f(g(x)) = sin(3x + 1)

def g(x):   return 3 * x + 1
def f(u):   return np.sin(u)
def dg(x):  return 3.0          # נגזרת של g
def df(u):  return np.cos(u)    # נגזרת של f

x0 = 1.0
u0 = g(x0)
y0 = f(u0)

# חוק השרשרת
dy_dx = df(u0) * dg(x0)

# אימות נומרי (הפרשים סופיים)
eps = 1e-7
dy_dx_numeric = (f(g(x0 + eps)) - f(g(x0 - eps))) / (2 * eps)

print('y = f(g(x)) = sin(3x + 1)')
print(f'x₀ = {x0},  u₀ = g(x₀) = {u0},  y₀ = {y0:.4f}')
print()
print('חוק השרשרת:')
print(f'  dy/du = f\' (u₀) = cos({u0}) = {df(u0):.4f}')
print(f'  du/dx = g\' (x₀) = {dg(x0)}')
print(f'  dy/dx = {df(u0):.4f} × {dg(x0)} = {dy_dx:.4f}')
print()
print(f'אימות נומרי: {dy_dx_numeric:.4f}  ← זהה! ✓')

<div dir=rtl style="text-align: right">

## הרשת שלנו — Mini-Network

נגדיר רשת פשוטה עם **3 פרמטרים**: $w, b, v$

$$z = w \cdot x + b \qquad \text{(שכבה לינארית)}$$
$$h = \text{ReLU}(z) = \max(0, z) \qquad \text{(אקטיבציה)}$$
$$\hat{y} = v \cdot h \qquad \text{(שכבה פלט)}$$
$$L = (y^* - \hat{y})^2 \qquad \text{(MSE Loss)}$$

**הגרף החישובי** — ויזואליזציה של הזרימה:

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlim(-0.5, 15)
ax.set_ylim(-0.5, 6)
ax.axis('off')
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('#FAFAFA')

def op_box(cx, cy, line1, line2, color, bw=2.2, bh=1.2):
    rect = mpatches.FancyBboxPatch((cx-bw/2, cy-bh/2), bw, bh,
                                   boxstyle='round,pad=0.12',
                                   facecolor=color, edgecolor='#2C3E50',
                                   linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(cx, cy+0.22, line1, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white', zorder=4)
    ax.text(cx, cy-0.22, line2, ha='center', va='center', fontsize=11,
            color='#FFE082', zorder=4, fontweight='bold')

def inp_node(cx, cy, name, color='#546E7A'):
    c = mpatches.Circle((cx, cy), 0.42, facecolor=color,
                         edgecolor='#2C3E50', linewidth=2, zorder=3)
    ax.add_patch(c)
    ax.text(cx, cy, name, ha='center', va='center', fontsize=11,
            fontweight='bold', color='white', zorder=4)

def arr(x1, y1, x2, y2, color='#37474F', lw=2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw), zorder=2)

# Input nodes
inp_node(0.7, 5.0, 'x', '#1565C0')
inp_node(0.7, 3.5, 'w', '#1565C0')
inp_node(0.7, 2.0, 'b', '#1565C0')
inp_node(7.5, 1.5, 'v', '#1565C0')
inp_node(11.5, 5.2, 'y*', '#1565C0')

# Operation nodes
op_box(3.5, 3.5, 'wx + b', 'z', '#1565C0')
op_box(7.0, 3.5, 'ReLU(z)', 'h', '#6A1B9A')
op_box(10.5, 3.5, 'v · h', 'ŷ', '#2E7D32')
op_box(13.5, 3.5, '(y*−ŷ)²', 'L', '#C62828')

# Forward arrows
arr(1.12, 4.8, 2.5, 3.9)
arr(1.12, 3.5, 2.5, 3.5)
arr(1.12, 2.2, 2.5, 3.1)
arr(4.6,  3.5, 6.0, 3.5)
arr(8.1,  3.5, 9.45, 3.5)
arr(7.92, 1.9, 9.45, 3.1)
arr(11.55,3.5, 12.45, 3.5)
arr(11.92,5.0, 12.8, 3.9)

# Backward gradient arrows
for x1, x2, y, label in [(12.45,11.55,4.1,'∂L/∂ŷ'), (9.45,8.1,4.1,'∂L/∂h'), (6.0,4.6,4.1,'∂L/∂z')]:
    arr(x1, y, x2, y, color='#E53935', lw=2.5)
    ax.text((x1+x2)/2, y+0.35, label, ha='center', fontsize=9,
            color='#E53935', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFEBEE',
                      edgecolor='#E53935', linewidth=1))

# Final gradients
for cx, label in [(0.7, '∂L/∂w, ∂L/∂b'), (7.5, '∂L/∂v')]:
    ax.text(cx, 0.5, label, ha='center', fontsize=9, color='#E53935',
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                      edgecolor='#E53935', linewidth=1.5))

ax.set_title('הגרף החישובי — כחול: מעבר קדימה | אדום: מעבר אחורה',
             fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

<div dir=rtl style="text-align: right">

## מעבר קדימה — Forward Pass

### הרעיון: שמרו הכל!

במהלך המעבר קדימה, כל פעולה **מחשבת את פלטה ושומרת את קלטיה**.

ה-**cache** הזה הוא קריטי — בלעדיו לא נוכל לחשב גרדיאנטים.

**למה?** כי הנגזרת של כל פעולה **תלויה** בערכי הכניסה שלה:

| פעולה | נגזרת — תלויה ב... |
|--------|--------------------|
| $z = wx + b$ | $\partial z/\partial w = x$ — צריכים את $x$ |
| $h = \text{ReLU}(z)$ | $\partial h/\partial z = \mathbb{1}[z > 0]$ — צריכים את $z$ |
| $\hat{y} = v \cdot h$ | $\partial \hat{y}/\partial v = h$ — צריכים את $h$ |

</div>

In [ ]:
def forward_pass(w, b, v, x, y_true, verbose=True):
    cache = {'w': w, 'b': b, 'v': v, 'x': x, 'y_true': y_true}

    if verbose:
        print('══════════ מעבר קדימה ══════════')

    z = w * x + b
    cache['z'] = z
    if verbose:
        print(f'  z = w·x + b  =  {w:.3f} × {x:.3f} + {b:.3f}  =  {z:.4f}')

    h = max(0.0, z)
    cache['h'] = h
    if verbose:
        status = 'פעיל ✓' if z > 0 else 'חסום ✗ (z≤0)'
        print(f'  h = ReLU(z)  =  max(0, {z:.4f})  =  {h:.4f}   [{status}]')

    y_hat = v * h
    cache['y_hat'] = y_hat
    if verbose:
        print(f'  ŷ = v·h      =  {v:.3f} × {h:.4f}  =  {y_hat:.4f}')

    L = (y_true - y_hat) ** 2
    cache['L'] = L
    if verbose:
        print(f'  L = (y*-ŷ)²  =  ({y_true:.3f} - {y_hat:.4f})²  =  {L:.4f}')
        print(f'  ┌── Cache נשמר: z={z:.3f}, h={h:.3f}, y_hat={y_hat:.3f}')

    return L, cache


# ריצה לדוגמה
w, b, v, x, y_true = 0.5, -0.3, 2.0, 1.5, 3.0
print(f'פרמטרים: w={w}, b={b}, v={v}')
print(f'קלט: x={x}, יעד: y*={y_true}\n')
L, cache = forward_pass(w, b, v, x, y_true)

<div dir=rtl style="text-align: right">

## מעבר אחורה — Backward Pass

### הרעיון: חוק השרשרת בפעולה

מתחילים מה-Loss ומתקדמים **לאחור**, שלב אחר שלב.

בכל צומת: מקבלים גרדיאנט מהצומת שאחריו, **מכפילים בגרדיאנט המקומי**, ומעבירים הלאה.

$$\frac{\partial L}{\partial w} = \underbrace{\frac{\partial L}{\partial \hat{y}}}_{\text{מגיע מאחור}} \cdot \underbrace{\frac{\partial \hat{y}}{\partial h}}_{\text{מקומי}} \cdot \underbrace{\frac{\partial h}{\partial z}}_{\text{מקומי}} \cdot \underbrace{\frac{\partial z}{\partial w}}_{\text{מקומי}}$$

**כל צומת צריך לדעת רק שתי דברים:**
1. הגרדיאנט שמגיע אליו מ'ימין' (מהצאצא)
2. הגרדיאנט המקומי שלו — שחישבנו מה-cache

</div>

In [ ]:
def backward_pass(cache, verbose=True):
    w, b, v = cache['w'], cache['b'], cache['v']
    x, z, h = cache['x'], cache['z'], cache['h']
    y_hat, y_true = cache['y_hat'], cache['y_true']

    if verbose:
        print('══════════ מעבר אחורה ══════════')
        print('(מתחילים מה-Loss, מתקדמים שמאלה)')
        print()

    # שלב 1: נגזרת ה-Loss
    dL_dyhat = -2 * (y_true - y_hat)
    if verbose:
        print(f'  ∂L/∂ŷ  = -2(y*-ŷ)        = -2·({y_true:.3f}-{y_hat:.4f}) = {dL_dyhat:.4f}')
        print()

    # שלב 2: נגזרת לפי v (ŷ = v·h, ∂ŷ/∂v = h)
    dL_dv = dL_dyhat * h
    if verbose:
        print(f'  ∂L/∂v  = ∂L/∂ŷ · h        = {dL_dyhat:.4f} × {h:.4f} = {dL_dv:.4f}')

    # שלב 3: גרדיאנט דרך h (ŷ = v·h, ∂ŷ/∂h = v)
    dL_dh = dL_dyhat * v
    if verbose:
        print(f'  ∂L/∂h  = ∂L/∂ŷ · v        = {dL_dyhat:.4f} × {v:.3f}  = {dL_dh:.4f}')
        print()

    # שלב 4: גרדיאנט דרך ReLU (h = ReLU(z), ∂h/∂z = 1 if z>0 else 0)
    relu_prime = 1.0 if z > 0 else 0.0
    dL_dz = dL_dh * relu_prime
    if verbose:
        relu_str = f'1 (z={z:.3f}>0, פעיל)' if z > 0 else f'0 (z={z:.3f}≤0, חסום!)'
        print(f"  ∂L/∂z  = ∂L/∂h · ReLU'(z) = {dL_dh:.4f} × {relu_str} = {dL_dz:.4f}")
        print()

    # שלב 5: נגזרת לפי w (z = wx+b, ∂z/∂w = x)
    dL_dw = dL_dz * x
    if verbose:
        print(f'  ∂L/∂w  = ∂L/∂z · x        = {dL_dz:.4f} × {x:.3f}  = {dL_dw:.4f}')

    # שלב 6: נגזרת לפי b (z = wx+b, ∂z/∂b = 1)
    dL_db = dL_dz * 1.0
    if verbose:
        print(f'  ∂L/∂b  = ∂L/∂z · 1        = {dL_dz:.4f}')
        print()
        print(f'  ══ גרדיאנטים סופיים ══')
        print(f'  ∂L/∂w = {dL_dw:.4f}   ∂L/∂b = {dL_db:.4f}   ∂L/∂v = {dL_dv:.4f}')

    return {'w': dL_dw, 'b': dL_db, 'v': dL_dv}


grads = backward_pass(cache)

<div dir=rtl style="text-align: right">

## אימות — Numerical Gradient Check

איך נדע שהגרדיאנטים שחישבנו נכונים?

**בדיקה נומרית:** הגדרת הנגזרת מחדו"א:

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$$

זה **איטי** (עבור n פרמטרים דורש n·2 מעברים קדימה), אבל **פשוט ונכון** — מצוין לאימות.

Backprop לעומת זאת דורש **מעבר קדימה אחד + מעבר אחורה אחד** — ללא תלות במספר הפרמטרים!

</div>

In [ ]:
def numerical_grad(param_name, w, b, v, x, y_true, eps=1e-5):
    params = {'w': w, 'b': b, 'v': v}

    params[param_name] += eps
    L_plus, _ = forward_pass(params['w'], params['b'], params['v'], x, y_true, verbose=False)

    params[param_name] -= 2 * eps
    L_minus, _ = forward_pass(params['w'], params['b'], params['v'], x, y_true, verbose=False)

    return (L_plus - L_minus) / (2 * eps)


print('══ Gradient Check ══')
print(f'{"פרמטר":^8} {"Backprop (אנליטי)":^20} {"נומרי":^20} {"שגיאה":^15} {"✓?":^6}')
print('-' * 73)

for param in ['w', 'b', 'v']:
    analytical = grads[param]
    numerical  = numerical_grad(param, w, b, v, x, y_true)
    error = abs(analytical - numerical)
    ok = '✓' if error < 1e-6 else '✗'
    print(f'  {param:^6}  {analytical:^20.6f}  {numerical:^20.6f}  {error:^15.2e}  {ok:^4}')

print()
print('הגרדיאנטים זהים — Backprop עובד! ✓')

<div dir=rtl style="text-align: right">

## הדגמה אינטראקטיבית

כעת תוכלו לשחק עם הפרמטרים ולראות בזמן אמת:
- **ערכי המעבר קדימה** בכל צומת
- **גרדיאנטים** על כל קשת (מה Backprop מחשב)
- **צעד GD אחד** עם lr=0.1 — לאן הפרמטרים זזים

**דברים מעניינים לנסות:**
- שנו `w` ו-`b` כך ש-`z` יהיה שלילי — מה קורה לגרדיאנטים? (ReLU חוסם!)
- שנו `y*` (המטרה) — לאיזה כיוון כל פרמטר זזה?
- מה קורה כשה-Loss קרוב ל-0?

</div>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from bidi.algorithm import get_display
    _h = get_display
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-bidi', '-q'])
    from bidi.algorithm import get_display
    _h = get_display


# ── visualization ─────────────────────────────────────────────────────────────
def draw_step(step, w, b, v, x, y_true):

    z        = w * x + b
    h        = max(0.0, z)
    y_hat    = v * h
    L        = (y_true - y_hat) ** 2
    dL_dyhat = -2 * (y_true - y_hat)
    dL_dv    = dL_dyhat * h
    dL_dh    = dL_dyhat * v
    relu_p   = 1.0 if z > 0 else 0.0
    dL_dz    = dL_dh * relu_p
    dL_dw    = dL_dz * x
    dL_db    = dL_dz

    STEPS = [
      { 'phase':'init',     'glow':None,
        'title': _h('מצב התחלתי — ערכי קלט ופרמטרים'),
        'lines': [_h('ערכי קלט ופרמטרים:'),
                  f'   x = {x:.3f}  ,  y* = {y_true:.3f}',
                  f'   w = {w:.3f}  ,  b = {b:.3f}  ,  v = {v:.3f}',
                  '', _h('עוד לא חישבנו כלום.'), _h('לחץ על ▶ הבא כדי להתחיל.')],
        'fw':set(), 'bw':set() },

      { 'phase':'forward',  'glow':'z',
        'title': _h('קדימה שלב 1  —  z = wx + b'),
        'lines': ['z  =  w · x  +  b',
                  f'   =  {w:.3f} × {x:.3f}  +  {b:.3f}',
                  f'   =  {w*x:.4f}  +  ({b:.3f})',
                  f'   =  {z:.4f}',
                  '', _h(f'✓  נשמר ב-cache:  z = {z:.4f}')],
        'fw':{'z'}, 'bw':set() },

      { 'phase':'forward',  'glow':'h',
        'title': _h('קדימה שלב 2  —  h = ReLU(z)'),
        'lines': ['h  =  ReLU(z)  =  max(0, z)',
                  f'   =  max(0,  {z:.4f})',
                  f'   =  {h:.4f}',
                  '',
                  (_h('ReLU פעיל ✓  — z>0, גרדיאנט יעבור') if z>0
                   else _h('⚠ ReLU חסום! z≤0  →  h=0, גרדיאנט=0')),
                  _h(f'✓  נשמר ב-cache:  h = {h:.4f}')],
        'fw':{'z','h'}, 'bw':set() },

      { 'phase':'forward',  'glow':'yhat',
        'title': _h('קדימה שלב 3  —  ŷ = v · h'),
        'lines': ['ŷ  =  v · h',
                  f'   =  {v:.3f}  ×  {h:.4f}',
                  f'   =  {y_hat:.4f}',
                  '', _h(f'✓  נשמר ב-cache:  ŷ = {y_hat:.4f}')],
        'fw':{'z','h','yhat'}, 'bw':set() },

      { 'phase':'forward',  'glow':'L',
        'title': _h('קדימה שלב 4  —  L = (y*−ŷ)²'),
        'lines': ['L  =  (y*  −  ŷ)²',
                  f'   =  ({y_true:.3f}  −  {y_hat:.4f})²',
                  f'   =  ({y_true-y_hat:.4f})²',
                  f'   =  {L:.4f}',
                  '', _h('✓ מעבר קדימה הסתיים! מתחיל מעבר אחורה ←')],
        'fw':{'z','h','yhat','L'}, 'bw':set() },

      { 'phase':'backward', 'glow':'dL_dyhat',
        'title': _h('אחורה שלב 1  —  ∂L/∂ŷ'),
        'lines': ['∂L/∂ŷ  =  −2 · (y*  −  ŷ)',
                  f'       =  −2 × ({y_true:.3f}  −  {y_hat:.4f})',
                  f'       =  −2 × ({y_true-y_hat:.4f})',
                  f'       =  {dL_dyhat:.4f}',
                  '', _h('זהו הגרדיאנט הראשוני. ממנו נשרשר לאחור.')],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat'} },

      { 'phase':'backward', 'glow':'dL_dv',
        'title': _h('אחורה שלב 2  —  ∂L/∂v  ← פרמטר!'),
        'lines': ['∂L/∂v  =  ∂L/∂ŷ  ·  (∂ŷ/∂v)',
                  _h('       כי  ŷ=v·h  →  ∂ŷ/∂v = h'),
                  f'       =  {dL_dyhat:.4f}  ×  {h:.4f}   [h מה-cache]',
                  f'       =  {dL_dv:.4f}',
                  '', _h(f'★  גרדיאנט עבור  v  =  {dL_dv:.4f}')],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat','dL_dv'} },

      { 'phase':'backward', 'glow':'dL_dh',
        'title': _h('אחורה שלב 3  —  ∂L/∂h'),
        'lines': ['∂L/∂h  =  ∂L/∂ŷ  ·  (∂ŷ/∂h)',
                  _h('       כי  ŷ=v·h  →  ∂ŷ/∂h = v'),
                  f'       =  {dL_dyhat:.4f}  ×  {v:.3f}   [v מה-cache]',
                  f'       =  {dL_dh:.4f}',
                  '', _h('הגרדיאנט ממשיך אחורה לעבר z.')],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat','dL_dv','dL_dh'} },

      { 'phase':'backward', 'glow':'dL_dz',
        'title': _h('אחורה שלב 4  —  ∂L/∂z  (דרך ReLU)'),
        'lines': ["∂L/∂z  =  ∂L/∂h  ·  ReLU'(z)",
                  (_h(f"       ReLU'(z)=1  (z={z:.3f}>0, פעיל)") if z>0
                   else _h(f"       ReLU'(z)=0  (z={z:.3f}≤0, חסום!)")),
                  f'       =  {dL_dh:.4f}  ×  {relu_p:.0f}',
                  f'       =  {dL_dz:.4f}',
                  '',
                  (_h('גרדיאנט עובר דרך ReLU.') if z>0
                   else _h('⚠ גרדיאנט נחסם!  w,b לא יעודכנו.'))],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat','dL_dv','dL_dh','dL_dz'} },

      { 'phase':'backward', 'glow':'dL_dw',
        'title': _h('אחורה שלב 5  —  ∂L/∂w  ← פרמטר!'),
        'lines': ['∂L/∂w  =  ∂L/∂z  ·  (∂z/∂w)',
                  _h('       כי  z=wx+b  →  ∂z/∂w = x'),
                  f'       =  {dL_dz:.4f}  ×  {x:.3f}   [x מה-cache]',
                  f'       =  {dL_dw:.4f}',
                  '', _h(f'★  גרדיאנט עבור  w  =  {dL_dw:.4f}')],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat','dL_dv','dL_dh','dL_dz','dL_dw'} },

      { 'phase':'backward', 'glow':'dL_db',
        'title': _h('אחורה שלב 6  —  ∂L/∂b  ← פרמטר!'),
        'lines': ['∂L/∂b  =  ∂L/∂z  ·  (∂z/∂b)',
                  _h('       כי  z=wx+b  →  ∂z/∂b = 1'),
                  f'       =  {dL_dz:.4f}  ×  1',
                  f'       =  {dL_db:.4f}',
                  '', _h(f'★  גרדיאנט עבור  b  =  {dL_db:.4f}'),
                  _h('✓ כל הגרדיאנטים חושבו!')],
        'fw':{'z','h','yhat','L'}, 'bw':{'dL_dyhat','dL_dv','dL_dh','dL_dz','dL_dw','dL_db'} },
    ]

    s    = STEPS[step]
    fw   = s['fw']
    bw   = s['bw']
    glow = s['glow']

    fig = plt.figure(figsize=(17, 7.5))
    fig.patch.set_facecolor('#EEF2F7')
    ax  = fig.add_axes([0.01, 0.06, 0.56, 0.86])
    ax2 = fig.add_axes([0.60, 0.06, 0.38, 0.86])
    for a in (ax, ax2):
        a.axis('off'); a.set_facecolor('#EEF2F7')
    ax.set_xlim(-0.5, 15);  ax.set_ylim(-1.8, 6.8)
    ax2.set_xlim(0, 10);    ax2.set_ylim(0, 15)

    def glow_fx(cx, cy, bw2, bh2, circle=False):
        for off, al in [(0.22,0.35),(0.12,0.50)]:
            if circle:
                ax.add_patch(mpatches.Circle((cx,cy), 0.44+off,
                    facecolor='#FFD700', edgecolor='none', zorder=2, alpha=al))
            else:
                ax.add_patch(mpatches.FancyBboxPatch(
                    (cx-bw2/2-off, cy-bh2/2-off), bw2+2*off, bh2+2*off,
                    boxstyle='round,pad=0.1', facecolor='#FFD700',
                    edgecolor='none', zorder=2, alpha=al))

    def op_box(cx, cy, top, val_str, col, nid, bw2=2.2, bh2=1.2):
        act  = (glow == nid)
        done = nid in fw
        c  = col if (act or done) else '#90A4AE'
        ec = '#FFD700' if act else ('#2C3E50' if done else '#78909C')
        lw = 3.5       if act else (2.0       if done else 1.5)
        if act: glow_fx(cx, cy, bw2, bh2)
        ax.add_patch(mpatches.FancyBboxPatch((cx-bw2/2, cy-bh2/2), bw2, bh2,
            boxstyle='round,pad=0.12', facecolor=c, edgecolor=ec, linewidth=lw, zorder=3))
        tc = 'white'  if (act or done) else '#B0BEC5'
        vc = '#FFE082' if (act or done) else '#78909C'
        ax.text(cx, cy+0.22, top,     ha='center', va='center', fontsize=8.5, fontweight='bold', color=tc, zorder=4)
        ax.text(cx, cy-0.22, val_str, ha='center', va='center', fontsize=10,  fontweight='bold', color=vc, zorder=4)

    def inp_node(cx, cy, label, val, col='#1565C0'):
        act = (glow == label)
        if act: glow_fx(cx, cy, 0, 0, circle=True)
        ax.add_patch(mpatches.Circle((cx,cy), 0.44, facecolor=col,
            edgecolor='#FFD700' if act else '#2C3E50', linewidth=3 if act else 2, zorder=3))
        ax.text(cx, cy+0.13, label,        ha='center', va='center', fontsize=9, fontweight='bold', color='white', zorder=4)
        ax.text(cx, cy-0.17, f'{val:.2f}', ha='center', va='center', fontsize=8, color='#FFE082', zorder=4)

    def arr(x1,y1,x2,y2,col='#546E7A',lw=1.8):
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color=col, lw=lw), zorder=2)

    def badge(cx,cy,txt,fc,ec,tc,show):
        if not show: return
        ax.text(cx,cy,txt, ha='center',va='center', fontsize=8.5, color=tc,
                fontweight='bold', zorder=5,
                bbox=dict(boxstyle='round,pad=0.22', facecolor=fc, edgecolor=ec, linewidth=1.5, alpha=0.97))

    inp_node(0.7, 5.0, 'x',  x)
    inp_node(0.7, 3.5, 'w',  w)
    inp_node(0.7, 2.0, 'b',  b)
    inp_node(7.0, 1.2, 'v',  v)
    inp_node(11.5,5.6, 'y*', y_true)

    relu_col = '#6A1B9A' if z>0 else '#607D8B'
    op_box(3.5,  3.5, 'wx+b',    f'z={z:.3f}'    if 'z'    in fw else 'z=?',   '#1565C0', 'z')
    op_box(7.0,  3.5, 'ReLU(z)', f'h={h:.3f}'    if 'h'    in fw else 'h=?',   relu_col,  'h')
    op_box(10.5, 3.5, 'v·h',     f'ŷ={y_hat:.3f}'if 'yhat' in fw else 'ŷ=?',  '#2E7D32', 'yhat')
    op_box(13.8, 3.5, '(y*-ŷ)²', f'L={L:.2f}'   if 'L'    in fw else 'L=?',   '#C62828', 'L')

    for (x1,y1,x2,y2) in [(1.14,4.78,2.5,3.9),(1.14,3.5,2.5,3.5),(1.14,2.22,2.5,3.1),
                           (4.6,3.5,6.0,3.5),(8.1,3.5,9.45,3.5),(7.44,1.6,9.45,3.1),
                           (11.6,3.5,12.7,3.5),(12.06,5.35,13.1,4.1)]:
        arr(x1,y1,x2,y2)

    badge(5.3,  4.2, f'∂L/∂z={dL_dz:.3f}',    '#FFEBEE','#C62828','#C62828', 'dL_dz'    in bw)
    badge(8.8,  4.2, f'∂L/∂h={dL_dh:.3f}',    '#FFEBEE','#C62828','#C62828', 'dL_dh'    in bw)
    badge(12.15,4.2, f'∂L/∂ŷ={dL_dyhat:.3f}', '#FFEBEE','#C62828','#C62828', 'dL_dyhat' in bw)

    bw_arrows = {'dL_dz':(5.3,4.4,4.7,4.4),'dL_dh':(8.8,4.4,8.1,4.4),
                 'dL_dyhat':(12.15,4.4,11.55,4.4),
                 'dL_dv':(8.1,2.5,9.5,3.1),'dL_dw':(1.55,4.5,0.9,4.0),'dL_db':(1.55,2.5,0.9,2.0)}
    if glow in bw_arrows:
        arr(*bw_arrows[glow], col='#C62828', lw=2.5)

    badge(0.7, 0.8,  f'∂L/∂w={dL_dw:.4f}', '#E8F5E9','#2E7D32','#1B5E20', 'dL_dw' in bw)
    badge(0.7, 0.05, f'∂L/∂b={dL_db:.4f}', '#E8F5E9','#2E7D32','#1B5E20', 'dL_db' in bw)
    badge(7.0, 0.3,  f'∂L/∂v={dL_dv:.4f}', '#E8F5E9','#2E7D32','#1B5E20', 'dL_dv' in bw)

    ax.set_title(s['title'], fontsize=12, fontweight='bold', pad=8, color='#1A237E')

    # right panel
    PCOL = {'init':'#546E7A','forward':'#1565C0','backward':'#B71C1C'}[s['phase']]
    PTXT = {'init':_h('התחלה'),'forward':_h('▶  מעבר קדימה'),'backward':_h('◀  מעבר אחורה')}[s['phase']]

    ax2.add_patch(mpatches.Rectangle((0,13.2),10,1.6, facecolor=PCOL, edgecolor='none'))
    ax2.text(5,14.1, PTXT,             ha='center',va='center', fontsize=13, fontweight='bold', color='white')
    ax2.text(5,13.45,f'step {step}/10',ha='center',va='center', fontsize=9,  color='white', alpha=0.85)

    ax2.add_patch(mpatches.FancyBboxPatch((0.2,7.0),9.6,5.9,
        boxstyle='round,pad=0.15', facecolor='white', edgecolor=PCOL, linewidth=2.5))
    for i, ln in enumerate(s['lines']):
        yp = 12.5 - i*(5.4/max(len(s['lines']),1))
        bold  = any(c in ln for c in ('=','★','✓','⚠'))
        size  = 10.5 if ('=' in ln and i>0) else (11 if i==0 else 9.5)
        color = '#1A237E' if '=' in ln else ('#2E7D32' if '★' in ln else ('#C62828' if '⚠' in ln else '#424242'))
        ax2.text(0.5,yp,ln, ha='left',va='center', fontsize=size,
                 fontweight='bold' if bold else 'normal', color=color,
                 fontfamily='monospace' if '=' in ln else 'sans-serif')

    ax2.add_patch(mpatches.Rectangle((0.2,0.2),9.6,6.5, facecolor='#F5F5F5', edgecolor='#90A4AE', linewidth=1.2))
    ax2.text(5,6.4, _h('cache — ערכים שמורים'), ha='center',va='center', fontsize=9, fontweight='bold', color='#37474F')

    rows = []
    if 'z'        in fw: rows.append(('fw', f'z         = {z:.4f}'))
    if 'h'        in fw: rows.append(('fw', f'h         = {h:.4f}'))
    if 'yhat'     in fw: rows.append(('fw', f'y_hat     = {y_hat:.4f}'))
    if 'L'        in fw: rows.append(('fw', f'L         = {L:.4f}'))
    if 'dL_dyhat' in bw: rows.append(('bw', f'dL/dy_hat = {dL_dyhat:.4f}'))
    if 'dL_dv'    in bw: rows.append(('par',f'dL/dv     = {dL_dv:.4f}  ★'))
    if 'dL_dh'    in bw: rows.append(('bw', f'dL/dh     = {dL_dh:.4f}'))
    if 'dL_dz'    in bw: rows.append(('bw', f'dL/dz     = {dL_dz:.4f}'))
    if 'dL_dw'    in bw: rows.append(('par',f'dL/dw     = {dL_dw:.4f}  ★'))
    if 'dL_db'    in bw: rows.append(('par',f'dL/db     = {dL_db:.4f}  ★'))

    BG  = {'fw':'#E3F2FD','bw':'#FFF3E0','par':'#E8F5E9'}
    COL = {'fw':'#1565C0','bw':'#E65100','par':'#1B5E20'}
    for i,(kind,txt) in enumerate(rows):
        yp = 5.8 - i*0.58
        ax2.add_patch(mpatches.Rectangle((0.4,yp-0.26),9.2,0.52, facecolor=BG[kind], edgecolor='#CFD8DC', linewidth=0.8))
        ax2.text(0.7,yp,txt, ha='left',va='center', fontsize=8.8,
                 color=COL[kind], fontweight='bold' if kind=='par' else 'normal', fontfamily='monospace')
    if not rows:
        ax2.text(5,3.5, _h('ריק — טרם חושב כלום'), ha='center',va='center', fontsize=11, color='#90A4AE')

    plt.show()


# ── stateful widget setup ─────────────────────────────────────────────────────
state = {'step': 0}

# output area
out = widgets.Output()

# parameter sliders
sl_style  = {'description_width': '30px'}
sl_layout = widgets.Layout(width='210px')
w_sl  = widgets.FloatSlider(min=-3,max=3, step=0.1, value=0.5,  description='w',  style=sl_style, layout=sl_layout)
b_sl  = widgets.FloatSlider(min=-3,max=3, step=0.1, value=-0.3, description='b',  style=sl_style, layout=sl_layout)
v_sl  = widgets.FloatSlider(min=-3,max=3, step=0.1, value=2.0,  description='v',  style=sl_style, layout=sl_layout)
x_sl  = widgets.FloatSlider(min=-5,max=5, step=0.1, value=1.5,  description='x',  style=sl_style, layout=sl_layout)
yt_sl = widgets.FloatSlider(min=-5,max=5, step=0.1, value=3.0,  description='y*', style=sl_style, layout=sl_layout)

# step buttons
BTN = widgets.Layout(width='130px', height='40px')
btn_prev  = widgets.Button(description='◀ הקודם', button_style='info',    layout=BTN)
btn_next  = widgets.Button(description='הבא ▶',   button_style='success', layout=BTN)
btn_reset = widgets.Button(description='↺ איפוס', button_style='warning', layout=BTN)
step_lbl  = widgets.HTML(value='<b style="font-size:15px">שלב 0 / 10</b>',
                          layout=widgets.Layout(margin='auto 0 auto 15px'))

PHASE_NAMES = ['init','fw','fw','fw','fw','bw','bw','bw','bw','bw','bw']
STEP_LABELS = [
    'שלב 0: התחלה',
    'שלב 1 ▶ z=wx+b',   'שלב 2 ▶ h=ReLU(z)', 'שלב 3 ▶ ŷ=v·h', 'שלב 4 ▶ L=(y*-ŷ)²',
    'שלב 5 ◀ ∂L/∂ŷ',   'שלב 6 ◀ ∂L/∂v',    'שלב 7 ◀ ∂L/∂h',
    'שלב 8 ◀ ∂L/∂z',   'שלב 9 ◀ ∂L/∂w',    'שלב 10 ◀ ∂L/∂b',
]

def redraw():
    n = state['step']
    btn_prev.disabled  = (n == 0)
    btn_next.disabled  = (n == 10)
    step_lbl.value     = f'<b style="font-size:15px">{STEP_LABELS[n]}</b>'
    with out:
        clear_output(wait=True)
        draw_step(n, w_sl.value, b_sl.value, v_sl.value, x_sl.value, yt_sl.value)

def on_next(b):
    if state['step'] < 10: state['step'] += 1
    redraw()

def on_prev(b):
    if state['step'] > 0:  state['step'] -= 1
    redraw()

def on_reset(b):
    state['step'] = 0
    redraw()

def on_param_change(change):
    redraw()

btn_next.on_click(on_next)
btn_prev.on_click(on_prev)
btn_reset.on_click(on_reset)
for sl in [w_sl, b_sl, v_sl, x_sl, yt_sl]:
    sl.observe(on_param_change, names='value')

# layout & display
params_row = widgets.HBox([w_sl, b_sl, v_sl, x_sl, yt_sl],
                           layout=widgets.Layout(flex_wrap='wrap', margin='0 0 8px 0'))
btn_row    = widgets.HBox([btn_prev, btn_next, btn_reset, step_lbl],
                           layout=widgets.Layout(align_items='center', margin='0 0 10px 0'))

display(params_row, btn_row, out)
redraw()

<div dir=rtl style="text-align: right">

---

## סיכום

| מושג | תיאור |
|------|-------|
| **חוק השרשרת** | הבסיס המתמטי — גרדיאנטים מורכבים |
| **גרף חישובי** | ייצוג ויזואלי של הרשת כהרכבת פונקציות |
| **Forward Pass** | חישוב הפלט + שמירת ערכי ביניים ב-cache |
| **Backward Pass** | חישוב גרדיאנטים מהפלט לאחור דרך ה-cache |
| **יעילות** | מעבר קדימה אחד + מעבר אחורה אחד = כל הגרדיאנטים |

**הרעיון המרכזי של Backprop:**  
כל פעולה בגרף היא "קופסה שחורה" שיודעת שתי דברים:
1. לחשב את פלטה מהכניסה (forward)
2. לחשב את הגרדיאנט לפי הכניסה מהגרדיאנט שקיבלה (backward)

זה מה ש-PyTorch/TensorFlow עושים בדיוק — **כל** פעולה ממומשת עם שני המעברים הללו, ו-**autograd** מרכיב אותם אוטומטית.

**בשיעור הבא:** נעבור מרשת עם 3 פרמטרים סקלריים לרשת עם **מטריצות משקולות** — ונראה שהרעיון זהה לחלוטין.

</div>